# Import Required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Load Project Utilities & Initialize Notebook Wigedts

In [0]:
%run /Workspace/Users/ariefdwir16@gmail.com/fmcg-domain/scripts/consolidated_pipe/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# GET data from Databricks Volume

dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")
dbutils.widgets.text("volume", "source_system", "Volume")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
data_source = dbutils.widgets.get("data_source")

base_path = f"/Volumes/{catalog}/{schema}/{volume}/{data_source}/*.csv"

print(base_path)



# Read data (Bronze)

In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df)

In [0]:
df.printSchema()

## Create Table Customer (Bronze)

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{schema}.{data_source}")

# Read data (Silver)

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

## Transformation
#### 1. Drop duplicates

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
print("Rows before duplicates dropped:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after duplicates dropped:", df_silver.count())


#### 2. Trim spaces in customer name

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))) #trailing space
)


In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))) #trailing space
)

#### 3. Data quality fix: Correcting City Typos

In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyerbad': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=['city'])
    .withColumn(
        "city", 
        F.when(F.col("city").isNull(), None)
        .when(F.col("city").isin(allowed), F.col("city"))
        .otherwise(None)
    )
)

#sanity checks
df_silver.select("city").distinct().show()


#### 4. Fix Title-casing Issue

In [0]:
df_silver.select("customer_name").distinct().show()


In [0]:
#title case fix
df_silver = df_silver.withColumn(
    "customer_name", 
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap(F.col("customer_name")))
)

df_silver.select("customer_name").distinct().show()

#### 5. Handling missing cities

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ["Sprintx Nutrition ", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
# Business confitmation note: city corecttions confitmed by bussines team

customer_city_fix = {
    #Sprintx Nutrition
    789403: "New Delhi",
    #Zenathlete Foods
    789420: "Bengaluru",
    789421: "Hyderabad",
    # Primefuel Nutrition
    789521: "Hyderabad",
    # Recovery Lane
    789603: "Hyderabad"

}

df_fix = spark.createDataFrame(
    [(k, v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city", 
        F.coalesce("city", "fixed_city") #replace null with fixed city
    )
    .drop("fixed_city")
)

display(df_silver)

In [0]:
#sanity checks
null_customer_names = ["Sprintx Nutrition ", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

#### 6. Convert customer_id to string

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

## Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    #build final customer column: CustomerName-city or CustomerName-Unknown
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    #static atributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Read data (Gold)

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merging Data source with Parent

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

##